# Minimal 45-Cell Reproducibility Matrix

**Paper:** *Hand-Authored Rules Add Cost Without Adding Recognition* (AI4Ag @ NeurIPS)

This notebook runs the **Paper-2 matrix**: 5 backbones × 3 datasets × 3 folds = 45 cells.
Each cell trains for 30 epochs with a shared recipe (AdamW, cosine LR, label smoothing 0.1).

| Backbone | Params | Family |
|---|---|---|
| YOLOv8n-cls | 1.4M | YOLO |
| MobileNetV3-Small | 2.5M | torchvision |
| EfficientNet-B0 | 4.0M | torchvision |
| ResNet-50 | 25.6M | torchvision |
| ViT-B/16 | 86.6M | torchvision |

**Runtime:** ~12-18h on free T4, ~6h on A100. Auto-resumes on disconnect.

**Datasets:** PlantVillage (US), Rice Leaf BD (Bangladesh), Wheat Leaf BD (Bangladesh)

---

In [ ]:
#@title 1. Clone repo & install deps
import os

%cd /content
if not os.path.isdir('agri-drone'):
    !git clone https://github.com/Ashut0sh-mishra/agri-drone.git
%cd /content/agri-drone
!git fetch --all && git pull
!pip install -q ultralytics omegaconf pyyaml torchvision

In [ ]:
#@title 2. Verify GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU! Go to Runtime → Change runtime type → GPU'
gpu = torch.cuda.get_device_name(0)
print(f'\n✓ CUDA {torch.version.cuda} — {gpu}')
print(f'  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
#@title 3. Mount Google Drive (for crash-resumable results)
from google.colab import drive
drive.mount('/content/drive')

import pathlib
DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/agri-drone/results_paper2')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print(f'Results dir: {DRIVE_OUT}')

In [ ]:
#@title 4. Download datasets from Kaggle (with 6-deep fallbacks)
#@markdown Upload your `kaggle.json` to Colab, or set env vars.
#@markdown The download script tries the primary slug first, then
#@markdown each fallback until one succeeds.

import json, os

# Option A: Upload kaggle.json
kaggle_json = pathlib.Path('/root/.kaggle/kaggle.json')
if not kaggle_json.exists():
    # Try Colab secrets
    try:
        from google.colab import userdata
        uname = userdata.get('KAGGLE_USERNAME')
        key = userdata.get('KAGGLE_KEY')
        if uname and key:
            kaggle_json.parent.mkdir(parents=True, exist_ok=True)
            kaggle_json.write_text(json.dumps({'username': uname, 'key': key}))
            os.chmod(str(kaggle_json), 0o600)
            print(f'Created {kaggle_json} from Colab secrets')
    except Exception:
        pass

if not kaggle_json.exists():
    print('⚠ No kaggle.json found. Upload manually or set KAGGLE_USERNAME/KAGGLE_KEY in Colab secrets.')
    print('  Alternatively, place datasets in datasets/externals/ via Google Drive.')
else:
    print(f'✓ Kaggle credentials found at {kaggle_json}')

# Download each dataset with fallbacks
import yaml
with open('configs/matrix/paper2.yaml') as f:
    cfg = yaml.safe_load(f)

for ds_name, ds_spec in cfg.get('datasets', {}).items():
    primary = ds_spec.get('kaggle_slug', '')
    fallbacks = ds_spec.get('kaggle_fallbacks', [])
    slugs = [primary] + fallbacks
    
    out_dir = pathlib.Path(f'datasets/externals/{ds_name}')
    if out_dir.exists() and any(out_dir.iterdir()):
        print(f'✓ {ds_name}: already present at {out_dir}')
        continue
    
    downloaded = False
    for slug in slugs:
        if not slug:
            continue
        print(f'  Trying {slug} for {ds_name}...')
        ret = os.system(f'kaggle datasets download -d {slug} -p /tmp/kaggle_{ds_name} --unzip -q 2>/dev/null')
        if ret == 0:
            # Move to datasets/externals/
            src = pathlib.Path(f'/tmp/kaggle_{ds_name}')
            out_dir.mkdir(parents=True, exist_ok=True)
            for item in src.iterdir():
                import shutil
                dest = out_dir / item.name
                if item.is_dir():
                    shutil.copytree(str(item), str(dest), dirs_exist_ok=True)
                else:
                    shutil.copy2(str(item), str(dest))
            print(f'  ✓ {ds_name}: downloaded from {slug}')
            downloaded = True
            break
    
    if not downloaded:
        print(f'  ✗ {ds_name}: all {len(slugs)} slugs failed. Place data manually.')

In [ ]:
#@title 5. Run the 45-cell matrix (crash-resumable)
#@markdown This is the main training loop. On disconnect, re-run this
#@markdown cell — it reads per_run.jsonl and skips completed cells.

import subprocess, sys, time

MATRIX_CONFIG = 'configs/matrix/paper2.yaml'
OUT_DIR = str(DRIVE_OUT / 'matrix')

# Symlink local results to Drive for persistence
local_out = pathlib.Path('evaluate/results/v2/matrix')
local_out.parent.mkdir(parents=True, exist_ok=True)
if not local_out.exists():
    local_out.symlink_to(OUT_DIR)

start = time.time()
print(f'Starting matrix run: {MATRIX_CONFIG}')
print(f'Output: {OUT_DIR}')
print(f'Resume: reads per_run.jsonl, skips status=ok cells\n')

# Stream output in real-time
proc = subprocess.Popen(
    [sys.executable, 'evaluate/matrix/run_matrix.py',
     '--config', MATRIX_CONFIG,
     '--out-dir', OUT_DIR,
     '--tracker', 'none'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
elapsed = time.time() - start
print(f'\n{"="*60}')
print(f'Matrix completed in {elapsed/3600:.1f}h (exit code {proc.returncode})')

In [ ]:
#@title 6. Post-training statistical analysis

# McNemar + Bootstrap CIs on the main ablation
!python evaluate/fix_statistics.py

# V2 statistical tests (per-class bootstrap, Holm-Bonferroni)
!python evaluate/statistical_tests.py --v2 --n-boot 10000 || echo 'v2 stats: needs scipy'

# EML sensitivity sweep
!python evaluate/eml_sensitivity.py || echo 'EML sensitivity: needs eml_analysis'

# Derive crossover ratio
!python evaluate/derive_crossover_ratio.py || echo 'Crossover: needs eml_comparison.csv'

In [ ]:
#@title 7. Aggregate results from per_run.jsonl

import json, pathlib
import pandas as pd

jsonl = DRIVE_OUT / 'matrix' / 'per_run.jsonl'
if not jsonl.exists():
    print('No per_run.jsonl found. Run cell 5 first.')
else:
    records = []
    for line in jsonl.read_text().strip().splitlines():
        rec = json.loads(line)
        if rec.get('status') == 'ok':
            records.append({
                'backbone': rec.get('backbone', ''),
                'dataset': rec.get('dataset', ''),
                'fold': rec.get('fold', 0),
                'seed': rec.get('seed', 0),
                'accuracy': rec.get('accuracy', 0),
                'macro_f1': rec.get('macro_f1', 0),
                'latency_ms': rec.get('latency_ms', 0),
                'train_time_s': rec.get('train_time_s', 0),
            })
    
    if records:
        df = pd.DataFrame(records)
        print(f'\n{len(records)} / 45 cells completed\n')
        
        # Summary table: mean ± std across folds, grouped by backbone × dataset
        summary = df.groupby(['backbone', 'dataset']).agg(
            acc_mean=('accuracy', 'mean'),
            acc_std=('accuracy', 'std'),
            f1_mean=('macro_f1', 'mean'),
            f1_std=('macro_f1', 'std'),
            lat_mean=('latency_ms', 'mean'),
        ).round(4)
        print(summary.to_string())
        
        # Save summary CSV
        summary.to_csv(str(DRIVE_OUT / 'matrix_summary.csv'))
        print(f'\nSaved: {DRIVE_OUT / "matrix_summary.csv"}')
    else:
        n_total = sum(1 for line in jsonl.read_text().strip().splitlines())
        print(f'0 cells with status=ok out of {n_total} total records.')
        print('Check per_run.jsonl for errors.')

In [ ]:
#@title 8. Archive results to Drive

import shutil, time

# Copy all evaluate/results/ to Drive for safekeeping
stamp = time.strftime('%Y%m%d_%H%M%S')
archive_name = f'/content/drive/MyDrive/agri-drone/results_paper2_{stamp}'
shutil.make_archive(archive_name, 'zip', 'evaluate/results')
print(f'Archived: {archive_name}.zip')

# Also copy the key JSON artefacts
key_files = [
    'evaluate/results/mcnemar_A_vs_B.json',
    'evaluate/results/ablation_summary.json',
    'evaluate/results/eml_summary.json',
    'evaluate/results/v2/statistics/holm_bonferroni_mcnemar.json',
]
for f in key_files:
    src = pathlib.Path(f)
    if src.exists():
        dst = DRIVE_OUT / src.name
        shutil.copy2(str(src), str(dst))
        print(f'  Copied {src.name} → {dst}')

print('\nAll results saved to Google Drive.')

---

## 48-Hour Priority Plan (4/10 → 7/10)

### Hour 0-6: Data & Splits
- Download all 3 datasets via cell 4 (or place on Drive)
- Verify group-aware splits: `python scripts/make_splits.py --input-dir data/raw/wheat --out-dir data/splits/wheat`
- Confirm no `aug_*` leakage in split manifest

### Hour 6-18: Matrix Training (cells 5-6)
- Run 45-cell matrix on free T4 (~12-18h)
- Colab may disconnect — re-run cell 5, it auto-resumes
- Monitor progress in per_run.jsonl

### Hour 18-24: Statistical Analysis
- Run cell 6 (McNemar, Bootstrap CIs, EML)
- Run cell 7 (aggregate matrix results)
- Verify: all 45 cells show status=ok in per_run.jsonl

### Hour 24-36: Paper Update
- Update Table 5.1 with 5-backbone results
- Add cross-backbone McNemar comparison
- Fill in §5.7 threats with actual numbers
- Add PDT caveat paragraph (already added)

### Hour 36-44: Code Cleanup
- Remove dry-run stubs from v2/matrix/
- Verify all cited artefacts exist and are non-empty
- Run `python evaluate/fix_statistics.py` for final n=934 correction

### Hour 44-48: Final Checks
- Run cell 8 (archive to Drive)
- `git add -A && git commit -m 'Paper-2 matrix results'`
- Verify claim traceability: every number in the paper maps to a file
- Read FINAL_REPO_AUDIT_FOR_EXTERNAL_REVIEW.md and check each item

### Score Impact (expected)
| Fix | Points |
|---|---|
| 3-layer defense (closed vocab fix) | +0.5 |
| Group-aware splits (leakage fix) | +0.5 |
| Real McNemar + Holm-Bonferroni | +0.5 |
| 45-cell matrix (real, not dry-run) | +1.0 |
| n=935→934 + PDT caveat | +0.3 |
| Paper dedup (8 archived) | +0.2 |
| **Total expected** | **+3.0 (4/10 → 7/10)** |